In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import random
import sqlite3
import datetime

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Load intents from JSON file
with open('intents.json', 'r') as file:
    intents = json.load(file)

# Database connection
conn = sqlite3.connect('college_database.db')
cursor = conn.cursor()

# Create necessary tables if they don't exist
cursor.execute('''CREATE TABLE IF NOT EXISTS courses
                  (id INTEGER PRIMARY KEY, name TEXT, description TEXT, credits INTEGER)''')
cursor.execute('''CREATE TABLE IF NOT EXISTS faculty
                  (id INTEGER PRIMARY KEY, name TEXT, department TEXT, email TEXT)''')
cursor.execute('''CREATE TABLE IF NOT EXISTS events
                  (id INTEGER PRIMARY KEY, name TEXT, date TEXT, location TEXT)''')
conn.commit()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

def get_intent(user_input):
    user_input = preprocess(user_input)
    tfidf = TfidfVectorizer().fit_transform([user_input] + [' '.join(intent['patterns']) for intent in intents['intents']])
    similarities = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    index = np.argmax(similarities)
    return intents['intents'][index] if similarities[index] > 0.3 else None

def get_course_info(course_name):
    cursor.execute("SELECT * FROM courses WHERE name LIKE ?", ('%' + course_name + '%',))
    course = cursor.fetchone()
    if course:
        return f"Course: {course[1]}\nDescription: {course[2]}\nCredits: {course[3]}"
    else:
        return "I couldn't find information about that course."

def get_faculty_info(faculty_name):
    cursor.execute("SELECT * FROM faculty WHERE name LIKE ?", ('%' + faculty_name + '%',))
    faculty = cursor.fetchone()
    if faculty:
        return f"Name: {faculty[1]}\nDepartment: {faculty[2]}\nEmail: {faculty[3]}"
    else:
        return "I couldn't find information about that faculty member."

def get_upcoming_events():
    current_date = datetime.datetime.now().strftime("%Y-%m-%d")
    cursor.execute("SELECT * FROM events WHERE date >= ? ORDER BY date LIMIT 5", (current_date,))
    events = cursor.fetchall()
    if events:
        return "\n".join([f"{event[1]} on {event[2]} at {event[3]}" for event in events])
    else:
        return "There are no upcoming events scheduled at the moment."

def handle_query(intent, user_input):
    if intent['tag'] == 'course_info':
        course_name = [word for word in user_input.split() if word not in stop_words][-1]
        return get_course_info(course_name)
    elif intent['tag'] == 'faculty_info':
        faculty_name = [word for word in user_input.split() if word not in stop_words][-1]
        return get_faculty_info(faculty_name)
    elif intent['tag'] == 'upcoming_events':
        return get_upcoming_events()
    else:
        return random.choice(intent['responses'])

def chatbot():
    print("College Chatbot: Hello! How can I assist you today?")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("College Chatbot: Goodbye! Have a great day!")
            break
        
        intent = get_intent(user_input)
        if intent:
            response = handle_query(intent, user_input)
        else:
            response = "I'm sorry, I don't understand. Could you please rephrase your question?"
        
        print("College Chatbot:", response)

if __name__ == "__main__":
    chatbot()

# Close the database connection when done
conn.close()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


College Chatbot: Hello! How can I assist you today?
